# limit_impact_v01：批量评估 D+1 涨跌停/停牌对拟合篮子的影响

本 notebook 参考 `correlation_v03.ipynb` 的任务结构，对 `construction_date_ls` 中每个构建日 D 独立执行完整流程：

1. 使用 D 日权重和收盘价构建 basket1；
2. 使用迅投历史数据识别 D+1 09:31 涨停、跌停、停牌与人工禁买；
3. basket2 直接清零 basket1 中不可购买的持仓，其他数量不变；
4. basket3 在全部指数成分股中排除不可购买股票后重新拟合；
5. 以 D+1 09:31 close 计算三个篮子的偏差，Task11 合并为一张 `3 * len(construction_date_ls)` 行的大表。

每个日期使用独立输出子目录，本流程不使用同花顺数据源。


## Task0：导入与指数配置


In [1]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from xtquant import xtdata

xtdata.enable_hello = False
xtdata.reconnect(port=58610)

from utils import gogoal_query, read_daily_data
from utils.limit_impact_v01 import (
    combine_task11_summaries,
    normalize_construction_date_list,
)
from utils.limit_impact_pipeline_v01 import (
    LimitImpactDateRun,
    LimitImpactPipelineConfig,
)

PROJECT_ROOT = Path.cwd().resolve()

INDEX_CODE = "000300"
INDEX_META = {
    "000016": {"name": "上证50", "xt": "000016.SH"},
    "000300": {"name": "沪深300", "xt": "000300.SH"},
    "000905": {"name": "中证500", "xt": "000905.SH"},
    "000852": {"name": "中证1000", "xt": "000852.SH"},
    "000688": {"name": "科创50", "xt": "000688.SH"},
}
if INDEX_CODE not in INDEX_META:
    raise ValueError(f"INDEX_META 未配置指数 {INDEX_CODE}")

index_name = INDEX_META[INDEX_CODE]["name"]
xt_index_code = INDEX_META[INDEX_CODE]["xt"]
print("项目目录:", PROJECT_ROOT)
print("指数:", INDEX_CODE, index_name)


项目目录: E:\Codex\系统\Stock-Index-Fitting
指数: 000300 沪深300


## Task1：运行参数

`construction_date_ls` 为篮子构建日 D 的列表；每个 D 的评估日自动取下一个上交所交易日。列表不能为空或包含重复日期。人工禁买名单对所有日期生效，可为空。


In [2]:
# 篮子构建日 D 列表，按此顺序输出比较结果。
construction_date_ls = ["20260701", "20260702", "20260703", "20260706", "20260707", "20260708", "20260709", "20260710"]

# 人工指定的不可购买股票；可为空，对所有日期共用。
manual_unavailable_codes = ["000001"]
# manual_unavailable_codes = []
# manual_unavailable_codes = ["600000", "000001.SZ"]

target_stock_value = 4_500_000.0
rule_file_path = "security_buy_rules.csv"

# 与 correlation_v03 一致的风险模型和优化参数。
risk_matrix_mode = "correlation"  # "covariance" 或 "correlation"
risk_lookback_days = 5
risk_half_life_days = 3.0
risk_price_col = "lastPrice"

pareto_risk_candidate_count = 10
pareto_amount_candidate_count = 10
pareto_beam_width = 20
pareto_max_rounds = 50
pareto_stale_rounds_to_stop = 3
pareto_legal_neighbor_steps = 3

allow_over_budget = True
max_over_budget_ratio = 1.005

# 迅投原始三角 tick 的既有 NAS 路径。
source_tick_root = Path(r"Z:\高频行情迅投\ticks")
price_validation_tolerance = 0.0001
limit_price_tolerance = 1e-6
include_suspended_as_unavailable = True

construction_date_ls = normalize_construction_date_list(
    construction_date_ls
)
print("待测试构建日:", construction_date_ls)


待测试构建日: ['20260701', '20260702', '20260703', '20260706', '20260707', '20260708', '20260709', '20260710']


## Task2：初始化批量运行与日期隔离目录

每个构建日在 `date_runs/<construction_date>/` 下保留原有的输入、状态、篮子和报表；批量 Task11 结果保存在本次运行根目录的 `04_reports/`。


In [3]:
started_at = datetime.now().astimezone()
import_time = started_at.strftime("%Y%m%d-%H%M%S")
batch_run_dir = (
    PROJECT_ROOT
    / "data"
    / INDEX_CODE
    / f"{import_time}_limit_impact_v01_batch"
)
batch_reports_dir = batch_run_dir / "04_reports"
batch_reports_dir.mkdir(parents=True, exist_ok=True)

pipeline_config = LimitImpactPipelineConfig(
    index_code=INDEX_CODE,
    index_name=index_name,
    xt_index_code=xt_index_code,
    target_stock_value=target_stock_value,
    rule_file_path=rule_file_path,
    manual_unavailable_codes=tuple(manual_unavailable_codes),
    risk_matrix_mode=risk_matrix_mode,
    risk_lookback_days=risk_lookback_days,
    risk_half_life_days=risk_half_life_days,
    risk_price_col=risk_price_col,
    pareto_risk_candidate_count=pareto_risk_candidate_count,
    pareto_amount_candidate_count=pareto_amount_candidate_count,
    pareto_beam_width=pareto_beam_width,
    pareto_max_rounds=pareto_max_rounds,
    pareto_stale_rounds_to_stop=pareto_stale_rounds_to_stop,
    pareto_legal_neighbor_steps=pareto_legal_neighbor_steps,
    allow_over_budget=allow_over_budget,
    max_over_budget_ratio=max_over_budget_ratio,
    source_tick_root=source_tick_root,
    price_validation_tolerance=price_validation_tolerance,
    limit_price_tolerance=limit_price_tolerance,
    include_suspended_as_unavailable=(
        include_suspended_as_unavailable
    ),
)

date_runs = [
    LimitImpactDateRun(
        construction_date=construction_date,
        config=pipeline_config,
        project_root=PROJECT_ROOT,
        batch_run_dir=batch_run_dir,
        import_time=import_time,
        xtdata_client=xtdata,
        gogoal_query_fn=gogoal_query,
        daily_loader=read_daily_data,
    )
    for construction_date in construction_date_ls
]

print("日期数量:", len(date_runs))
print("构建日 -> D+1:")
for run in date_runs:
    print(" ", run.construction_date, "->", run.evaluation_date)
print("批量输出目录:", batch_run_dir)


日期数量: 8
构建日 -> D+1:
  20260701 -> 20260702
  20260702 -> 20260703
  20260703 -> 20260706
  20260706 -> 20260707
  20260707 -> 20260708
  20260708 -> 20260709
  20260709 -> 20260710
  20260710 -> 20260713
批量输出目录: E:\Codex\系统\Stock-Index-Fitting\data\000300\20260806-143719_limit_impact_v01_batch


## Task3：逐日读取指数权重与交易规则


In [4]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task3 "
        f"{run.construction_date}"
    )
    run.load_inputs()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "component_count": len(run.stock_codes),
                "raw_weight_sum_pct": run.df_index_weights[
                    "raw_weight_pct"
                ].sum(),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task3 20260701
[2/8] Task3 20260702
[3/8] Task3 20260703
[4/8] Task3 20260706
[5/8] Task3 20260707
[6/8] Task3 20260708
[7/8] Task3 20260709
[8/8] Task3 20260710


,construction_date,evaluation_date,component_count,raw_weight_sum_pct
0,20260701,20260702,300,100.0
1,20260702,20260703,300,100.0
2,20260703,20260706,300,100.0
3,20260706,20260707,300,100.0
4,20260707,20260708,300,100.0
5,20260708,20260709,300,100.0
6,20260709,20260710,300,100.0
7,20260710,20260713,300,100.0


## Task4：逐日读取 D 日指数与成分股收盘价

Go-Goal 为构建价主源，迅投指数日线和 NAS 股票日线用于交叉校验。


In [5]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task4 "
        f"{run.construction_date}"
    )
    run.load_construction_prices()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "construction_index_close": (
                    run.construction_index_close
                ),
                "validated_stock_count": len(
                    run.df_market_snapshot
                ),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task4 20260701


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


成功读取日行情数据
  文件路径: Z:\高频行情迅投\日行情\2026\07\A_DAYBAR_20260701.pkl
  数据形状: (5528, 28)
[2/8] Task4 20260702


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


成功读取日行情数据
  文件路径: Z:\高频行情迅投\日行情\2026\07\A_DAYBAR_20260702.pkl
  数据形状: (5530, 28)
[3/8] Task4 20260703


...

正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


成功读取日行情数据
  文件路径: Z:\高频行情迅投\日行情\2026\07\A_DAYBAR_20260709.pkl
  数据形状: (5529, 28)
[8/8] Task4 20260710


正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


成功读取日行情数据
  文件路径: Z:\高频行情迅投\日行情\2026\07\A_DAYBAR_20260710.pkl
  数据形状: (5530, 28)


,construction_date,construction_index_close,validated_stock_count
0,20260701,4958.9773,300
1,20260702,4812.2957,300
2,20260703,4842.1737,300
3,20260706,4841.9980,300
4,20260707,4792.2624,300
5,20260708,4755.5338,300
6,20260709,4876.3125,300
7,20260710,4780.7867,300


## Task5：逐日构建理论组合与风险矩阵


In [6]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task5 "
        f"{run.construction_date}"
    )
    run.build_risk_model()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "requested_risk_dates": run.risk_dates,
                "used_risk_dates": run.used_risk_dates,
                "skipped_risk_dates": run.skipped_risk_dates,
                "risk_matrix_shape": run.risk_matrix.shape,
                "oas_shrinkage": run.risk_model.summary[
                    "oas_shrinkage"
                ],
            }
            for run in date_runs
        ]
    )
)


[1/8] Task5 20260701
[CACHE HIT] risk cache 20260625 20260625: basket_minute_wide_20260625_f2653c108899.pkl (3.99 MB)
[CACHE HIT] risk cache 20260626 20260626: basket_minute_wide_20260626_e075d94ff3c9.pkl (3.99 MB)
[CACHE HIT] risk cache 20260629 20260629: basket_minute_wide_20260629_2d4119158d06.pkl (3.99 MB)
[CACHE HIT] risk cache 20260630 20260630: basket_minute_wide_20260630_353f712e13f0.pkl (3.99 MB)
[CACHE HIT] risk cache 20260701 20260701: basket_minute_wide_20260701_bfeca5272447.pkl (3.99 MB)
[Risk fallback 20260629] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[Risk fallback 20260630] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[Risk fallback 20260701] 使用 lastClose 填充全天无有效 lastPrice 的股票，共 1 只：['688072.SH']
[2/8] Task5 20260702
[CACHE HIT] risk cache 20260626 20260626: basket_minute_wide_20260626_e075d94ff3c9.pkl (3.99 MB)
[CACHE HIT] risk cache 20260629 20260629: basket_minute_wide_20260629_2d4119158d06.pkl (3.99 MB)
[CACHE HIT] risk cache 20260630 2026063

,construction_date,requested_risk_dates,used_risk_dates,skipped_risk_dates,risk_matrix_shape,oas_shrinkage
0,20260701,"[20260625, 20260626, 20260629, 20260630, 20260...","[20260625, 20260626, 20260629, 20260630, 20260...",[],"(300, 300)",0.029701
1,20260702,"[20260626, 20260629, 20260630, 20260701, 20260...","[20260626, 20260629, 20260630, 20260701, 20260...",[],"(300, 300)",0.024132
2,20260703,"[20260629, 20260630, 20260701, 20260702, 20260...","[20260629, 20260630, 20260701, 20260702, 20260...",[],"(300, 300)",0.020576
3,20260706,"[20260630, 20260701, 20260702, 20260703, 20260...","[20260630, 20260701, 20260702, 20260703, 20260...",[],"(300, 300)",0.018401
4,20260707,"[20260701, 20260702, 20260703, 20260706, 20260...","[20260701, 20260702, 20260703, 20260706, 20260...",[],"(300, 300)",0.016044
5,20260708,"[20260702, 20260703, 20260706, 20260707, 20260...","[20260702, 20260703, 20260706, 20260707, 20260...",[],"(300, 300)",0.012864
6,20260709,"[20260703, 20260706, 20260707, 20260708, 20260...","[20260703, 20260706, 20260707, 20260708, 20260...",[],"(300, 300)",0.013993
7,20260710,"[20260706, 20260707, 20260708, 20260709, 20260...","[20260706, 20260707, 20260708, 20260709, 20260...",[],"(300, 300)",0.015036


## Task6：按原流程逐日构建 basket1


In [7]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task6 "
        f"{run.construction_date}"
    )
    run.build_basket1()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "basket1_invested_amount": (
                    run.basket1_invested_amount
                ),
                "basket1_held_stock_count": int(
                    (run.basket1["target_qty"] > 0).sum()
                ),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task6 20260701
[pareto basket1] round=1, generated=1,694, frontier=63, beam=20, new=63, best_amount=0.17424769, best_TE=555.572522%
[pareto basket1] round=5, generated=37,873, frontier=105, beam=20, new=105, best_amount=0.16816950, best_TE=500.710978%
[pareto basket1] round=10, generated=28,274, frontier=102, beam=20, new=101, best_amount=0.16370712, best_TE=482.737040%
[pareto basket1] round=15, generated=23,955, frontier=113, beam=20, new=107, best_amount=0.16130604, best_TE=468.558171%
[pareto basket1] round=20, generated=16,262, frontier=117, beam=20, new=86, best_amount=0.15892463, best_TE=461.518452%
[pareto basket1] round=25, generated=16,059, frontier=179, beam=20, new=92, best_amount=0.15803714, best_TE=452.984629%
[pareto basket1] round=30, generated=11,154, frontier=175, beam=20, new=54, best_amount=0.15803714, best_TE=449.460903%
[pareto basket1] round=35, generated=13,515, frontier=259, beam=20, new=112, best_amount=0.15741513, best_TE=441.794752%
[pareto basket1] ro

,construction_date,basket1_invested_amount,basket1_held_stock_count
0,20260701,4521875.50,276
1,20260702,4513538.72,277
2,20260703,4512884.34,279
3,20260706,4521557.45,278
4,20260707,4522455.38,279
5,20260708,4510942.56,277
6,20260709,4516928.66,279
7,20260710,4502900.81,281


## Task7：逐日读取 D+1 09:31 价格与迅投涨跌停/停牌状态


In [8]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task7 "
        f"{run.construction_date} -> {run.evaluation_date}"
    )
    run.load_d1_status()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "unavailable_component_count": len(
                    run.status_unavailable_codes
                ),
                "manual_not_in_universe": (
                    run.manual_not_in_universe
                ),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task7 20260701 -> 20260702
[CACHE HIT] D+1 opening cache 20260701 20260702: basket_minute_wide_20260702_ea54ee2142d4.pkl (3.99 MB)
[2/8] Task7 20260702 -> 20260703
[CACHE HIT] D+1 opening cache 20260702 20260703: basket_minute_wide_20260703_bfa50f229284.pkl (3.99 MB)
[3/8] Task7 20260703 -> 20260706
[CACHE HIT] D+1 opening cache 20260703 20260706: basket_minute_wide_20260706_b0ca6a949f2e.pkl (3.99 MB)
[4/8] Task7 20260706 -> 20260707
[CACHE HIT] D+1 opening cache 20260706 20260707: basket_minute_wide_20260707_14ca0077a818.pkl (3.99 MB)
[5/8] Task7 20260707 -> 20260708
[CACHE HIT] D+1 opening cache 20260707 20260708: basket_minute_wide_20260708_828a77b111aa.pkl (3.99 MB)
[6/8] Task7 20260708 -> 20260709
[CACHE HIT] D+1 opening cache 20260708 20260709: basket_minute_wide_20260709_a7ccd4bdc8b1.pkl (3.99 MB)
[7/8] Task7 20260709 -> 20260710
[CACHE HIT] D+1 opening cache 20260709 20260710: basket_minute_wide_20260710_7c4ba95815ad.pkl (3.99 MB)
[8/8] Task7 20260710 -> 20260713
[CACHE H

,construction_date,evaluation_date,unavailable_component_count,manual_not_in_universe
0,20260701,20260702,2,[]
1,20260702,20260703,3,[]
2,20260703,20260706,2,[]
3,20260706,20260707,2,[]
4,20260707,20260708,5,[]
5,20260708,20260709,2,[]
6,20260709,20260710,2,[]
7,20260710,20260713,1,[]


## Task8：逐日直接剔除得到 basket2


In [9]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task8 "
        f"{run.construction_date}"
    )
    run.build_basket2()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "basket2_held_stock_count": int(
                    (run.basket2["target_qty"] > 0).sum()
                ),
                "removed_stock_count": len(
                    run.df_removed_from_basket1
                ),
                "removed_build_amount": run.df_removed_from_basket1[
                    "removed_build_amount"
                ].sum(),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task8 20260701
[2/8] Task8 20260702
[3/8] Task8 20260703
[4/8] Task8 20260706
[5/8] Task8 20260707
[6/8] Task8 20260708
[7/8] Task8 20260709
[8/8] Task8 20260710


,construction_date,basket2_held_stock_count,removed_stock_count,removed_build_amount
0,20260701,275,1,17272.0
1,20260702,276,1,17476.0
2,20260703,278,1,16464.0
3,20260706,277,1,18900.0
4,20260707,275,4,50673.0
5,20260708,276,1,19080.0
6,20260709,278,1,17833.0
7,20260710,280,1,18810.0


## Task9：逐日在全部成分股中排除不可买后重新拟合 basket3


In [10]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task9 "
        f"{run.construction_date}"
    )
    run.build_basket3()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "basket3_invested_amount": run.basket3[
                    "target_market_value"
                ].sum(),
                "basket3_held_stock_count": int(
                    (run.basket3["target_qty"] > 0).sum()
                ),
                "excluded_component_count": len(
                    run.all_unavailable_codes
                ),
            }
            for run in date_runs
        ]
    )
)


[1/8] Task9 20260701
[pareto basket3] round=1, generated=1,955, frontier=68, beam=20, new=68, best_amount=0.18098229, best_TE=562.473602%
[pareto basket3] round=5, generated=43,385, frontier=95, beam=20, new=95, best_amount=0.17433378, best_TE=500.809160%
[pareto basket3] round=10, generated=26,967, frontier=114, beam=20, new=91, best_amount=0.16941284, best_TE=481.524393%
[pareto basket3] round=15, generated=21,871, frontier=97, beam=20, new=73, best_amount=0.16661823, best_TE=468.944677%
[pareto basket3] round=20, generated=17,345, frontier=158, beam=20, new=115, best_amount=0.16400146, best_TE=458.635457%
[pareto basket3] round=25, generated=17,701, frontier=183, beam=20, new=97, best_amount=0.16324194, best_TE=449.979047%
[pareto basket3] round=30, generated=17,019, frontier=158, beam=20, new=110, best_amount=0.16324194, best_TE=446.555019%
[pareto basket3] round=35, generated=15,533, frontier=342, beam=20, new=73, best_amount=0.16324194, best_TE=440.322218%
[pareto basket3] round=

,construction_date,basket3_invested_amount,basket3_held_stock_count,excluded_component_count
0,20260701,4518254.60,276,2
1,20260702,4509515.57,278,3
2,20260703,4521331.84,279,2
3,20260706,4503732.31,278,2
4,20260707,4519078.42,274,5
5,20260708,4508641.70,279,2
6,20260709,4500793.73,279,2
7,20260710,4501956.93,278,1


## Task10：逐日读取 D+1 指数 09:31 close


In [11]:
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task10 "
        f"{run.construction_date} -> {run.evaluation_date}"
    )
    run.load_index_opening_price()

display(
    pd.DataFrame(
        [
            {
                "construction_date": run.construction_date,
                "evaluation_date": run.evaluation_date,
                "construction_index_close": (
                    run.construction_index_close
                ),
                "opening_index_close": run.opening_index_close,
            }
            for run in date_runs
        ]
    )
)


[1/8] Task10 20260701 -> 20260702
[2/8] Task10 20260702 -> 20260703
[3/8] Task10 20260703 -> 20260706
[4/8] Task10 20260706 -> 20260707
[5/8] Task10 20260707 -> 20260708
[6/8] Task10 20260708 -> 20260709
[7/8] Task10 20260709 -> 20260710
[8/8] Task10 20260710 -> 20260713


,construction_date,evaluation_date,construction_index_close,opening_index_close
0,20260701,20260702,4958.9773,4864.983
1,20260702,20260703,4812.2957,4829.190
2,20260703,20260706,4842.1737,4855.503
3,20260706,20260707,4841.9980,4808.960
4,20260707,20260708,4792.2624,4798.470
5,20260708,20260709,4755.5338,4775.219
6,20260709,20260710,4876.3125,4885.546
7,20260710,20260713,4780.7867,4752.512


## Task11：汇总所有日期的三篮子 09:31 偏差

核心输出 `df_deviation_summary_all`。每个构建日固定按 basket1、basket2、basket3 输出三行；大表在原八列前增加 `construction_date`，共 `3 * len(construction_date_ls)` 行、9 列。


In [12]:
task11_summary_by_date = {}
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task11 "
        f"{run.construction_date}"
    )
    task11_summary_by_date[run.construction_date] = run.evaluate()

df_deviation_summary_all = combine_task11_summaries(
    task11_summary_by_date,
    construction_date_ls,
)
# 保留原单日 notebook 常用变量名，现在指向批量大表。
df_deviation_summary = df_deviation_summary_all

task11_report_path = (
    batch_reports_dir
    / "all_dates_three_basket_deviation_0931.csv"
)
df_deviation_summary_all.to_csv(
    task11_report_path,
    index=False,
    encoding="utf-8-sig",
)

assert len(df_deviation_summary_all) == 3 * len(
    construction_date_ls
)
assert (
    df_deviation_summary_all.groupby(
        "construction_date", sort=False
    ).size() == 3
).all()

print("Task11 批量报表:", task11_report_path)
display(df_deviation_summary_all)


[1/8] Task11 20260701
[2/8] Task11 20260702
[3/8] Task11 20260703
[4/8] Task11 20260706
[5/8] Task11 20260707
[6/8] Task11 20260708
[7/8] Task11 20260709
[8/8] Task11 20260710
Task11 批量报表: E:\Codex\系统\Stock-Index-Fitting\data\000300\20260806-143719_limit_impact_v01_batch\04_reports\all_dates_three_basket_deviation_0931.csv


,construction_date,basket,held_stock_count,basket_build_amount,basket_amount_0931,common_base_gap_amount,common_base_gap_pct,return_gap_pct,cash_adjusted_gap_pct
0,20260701,basket1,276,4521875.50,4436731.52,565.329924,0.012744,0.012502,0.012744
1,20260701,basket2,275,4504603.50,4419323.52,-16842.670076,-0.379667,0.002263,0.009678
2,20260701,basket3,276,4518254.60,4432335.02,-3831.170076,-0.086362,-0.006173,-0.004740
3,20260702,basket1,277,4513538.72,4529320.06,-64.127933,-0.001416,-0.001421,-0.001416
4,20260702,basket2,276,4496062.72,4511827.06,-17557.127933,-0.387627,-0.000440,-0.001791
5,20260702,basket3,278,4509515.57,4525365.94,-4018.247933,-0.088715,0.000422,0.000108
6,20260703,basket1,279,4512884.34,4524912.36,-394.828283,-0.008725,-0.008749,-0.008725
7,20260703,basket2,278,4496420.34,4508560.36,-16746.828283,-0.370071,-0.005282,-0.006250
8,20260703,basket3,279,4521331.84,4532315.71,7008.521717,0.154874,-0.032341,-0.031798
9,20260706,basket1,278,4521557.45,4494874.19,4168.304795,0.092821,0.092187,0.092821


## Task12：完整性检查与批量运行清单


In [13]:
run_manifests = []
for position, run in enumerate(date_runs, start=1):
    print(
        f"[{position}/{len(date_runs)}] Task12 "
        f"{run.construction_date}"
    )
    run_manifests.append(run.finalize())

batch_manifest = {
    "index_code": INDEX_CODE,
    "index_name": index_name,
    "started_at": started_at.isoformat(),
    "construction_date_ls": construction_date_ls,
    "evaluation_date_ls": [
        run.evaluation_date for run in date_runs
    ],
    "construction_date_count": len(construction_date_ls),
    "task11_row_count": len(df_deviation_summary_all),
    "task11_report_path": str(task11_report_path),
    "manual_unavailable_codes": (
        date_runs[0].manual_unavailable_codes
    ),
    "date_run_dirs": {
        run.construction_date: str(run.run_dir)
        for run in date_runs
    },
}
batch_manifest_path = batch_reports_dir / "batch_run_manifest.json"
with batch_manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(
        batch_manifest,
        handle,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

print("=" * 72)
print("limit_impact_v01 批量运行完成")
print("构建日数量:", len(construction_date_ls))
print("Task11 结果行数:", len(df_deviation_summary_all))
print("批量输出目录:", batch_run_dir)
print("=" * 72)
display(
    pd.DataFrame(
        [
            {
                "construction_date": manifest[
                    "construction_date"
                ],
                "evaluation_date": manifest["evaluation_date"],
                "unavailable_component_count": len(
                    manifest["status_unavailable_codes"]
                ),
                "removed_from_basket1_count": len(
                    manifest["removed_from_basket1_codes"]
                ),
                "run_dir": manifest["run_dir"],
            }
            for manifest in run_manifests
        ]
    )
)


[1/8] Task12 20260701
[2/8] Task12 20260702
[3/8] Task12 20260703
[4/8] Task12 20260706
[5/8] Task12 20260707
[6/8] Task12 20260708
[7/8] Task12 20260709
[8/8] Task12 20260710
limit_impact_v01 批量运行完成
构建日数量: 8
Task11 结果行数: 24
批量输出目录: E:\Codex\系统\Stock-Index-Fitting\data\000300\20260806-143719_limit_impact_v01_batch


,construction_date,evaluation_date,unavailable_component_count,removed_from_basket1_count,run_dir
0,20260701,20260702,2,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
1,20260702,20260703,3,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
2,20260703,20260706,2,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
3,20260706,20260707,2,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
4,20260707,20260708,5,4,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
5,20260708,20260709,2,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
6,20260709,20260710,2,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...
7,20260710,20260713,1,1,E:\Codex\系统\Stock-Index-Fitting\data\000300\20...


In [15]:
import pandas as pd

required_plot_columns = {
    "construction_date",
    "basket",
    "common_base_gap_pct",
    "common_base_gap_amount",
    "return_gap_pct",
    "cash_adjusted_gap_pct",
}
missing_columns = required_plot_columns - set(df_deviation_summary_all.columns)
if missing_columns:
    raise ValueError(f"绘图数据缺少字段: {sorted(missing_columns)}")

basket_order = ["basket1", "basket2", "basket3"]
basket_colors = {
    "basket1": "#1f77b4",
    "basket2": "#ff7f0e",
    "basket3": "#2ca02c",
}

plot_df = df_deviation_summary_all.copy()
plot_df["construction_date"] = pd.to_datetime(
    plot_df["construction_date"].astype(str),
    format="%Y%m%d",
)
plot_df["basket"] = pd.Categorical(
    plot_df["basket"],
    categories=basket_order,
    ordered=True,
)

plot_df = (
    plot_df.sort_values(["construction_date", "basket"])
    .reset_index(drop=True)
)

if plot_df[list(required_plot_columns)].isna().any(axis=None):
    bad_rows = plot_df.loc[
        plot_df[list(required_plot_columns)].isna().any(axis=1)
    ]
    display(bad_rows)
    raise ValueError("绘图数据中存在缺失值")

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_metric_plotly(
    data,
    metric,
    title,
    *,
    secondary_amount=False,
):
    fig = make_subplots(
        specs=[[{"secondary_y": secondary_amount}]]
    )

    # 主轴：三个篮子的百分比指标
    for basket in basket_order:
        basket_data = data.loc[data["basket"] == basket]

        fig.add_trace(
            go.Scatter(
                x=basket_data["construction_date"],
                y=basket_data[metric],
                mode="lines+markers",
                name=f"{basket} - {metric}",
                line={
                    "color": basket_colors[basket],
                    "width": 2.5,
                },
                marker={"size": 7},
                hovertemplate=(
                    "%{x|%Y-%m-%d}"
                    f"<br>{basket}"
                    f"<br>{metric}: %{{y:.6f}}%"
                    "<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # 第一张图副轴：三个篮子的 common_base_gap_amount
    if secondary_amount:
        for basket in basket_order:
            basket_data = data.loc[data["basket"] == basket]

            fig.add_trace(
                go.Scatter(
                    x=basket_data["construction_date"],
                    y=basket_data["common_base_gap_amount"],
                    mode="lines+markers",
                    name=f"{basket} - common_base_gap_amount",
                    line={
                        "color": basket_colors[basket],
                        "width": 1.8,
                        "dash": "dash",
                    },
                    marker={
                        "size": 7,
                        "symbol": "x",
                    },
                    opacity=0.75,
                    hovertemplate=(
                        "%{x|%Y-%m-%d}"
                        f"<br>{basket}"
                        "<br>common_base_gap_amount: "
                        "%{y:,.2f}"
                        "<extra></extra>"
                    ),
                ),
                secondary_y=True,
            )

    fig.add_hline(
        y=0,
        line_width=1,
        line_dash="dot",
        line_color="black",
        secondary_y=False,
    )

    fig.update_layout(
        title=title,
        xaxis_title="Construction Date",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"t": 120},
    )

    fig.update_xaxes(
        tickformat="%Y-%m-%d",
        showgrid=True,
    )
    fig.update_yaxes(
        title_text=f"{metric}（%）",
        ticksuffix="%",
        showgrid=True,
        secondary_y=False,
    )

    if secondary_amount:
        fig.update_yaxes(
            title_text="common_base_gap_amount",
            tickformat=",.0f",
            showgrid=False,
            secondary_y=True,
        )

    fig.show()
    return fig


plotly_figures = {
    "common_base_gap": plot_metric_plotly(
        plot_df,
        "common_base_gap_pct",
        "Basket Common Base Gap",
        secondary_amount=True,
    ),
    "return_gap": plot_metric_plotly(
        plot_df,
        "return_gap_pct",
        "Basket Return Gap",
    ),
    "cash_adjusted_gap": plot_metric_plotly(
        plot_df,
        "cash_adjusted_gap_pct",
        "Basket Cash-adjusted Gap",
    ),
}